# AntSDR E200 — PYNQ TX/RX loopback demo

Confirms that PYNQ on the E200 can drive the AD9361 transmitter and read the
receiver back, using a **physical TX → 10 dB attenuator → RX loopback**.

What it does:

1. Shows PYNQ sees the loaded PL design (`base.bit`).
2. Configures the AD9361 through `libiio` / `pyadi-iio`.
3. Transmits a single complex tone at a known baseband offset.
4. Captures the receiver and locates that tone in the spectrum.
5. Checks the received level is consistent with the 10 dB pad.

> **TX and RX must share the same LO.** The device-tree defaults are RX 2.4 GHz
> and TX 2.45 GHz. Left alone, a baseband tone would land 50 MHz away at the
> receiver — far outside the analysis bandwidth — and the demo would show
> nothing. Cell 3 sets both explicitly.

## 1. PYNQ sees the PL design

In [ ]:
from pynq import PL, Overlay
import os

BIT = "/home/xilinx/jupyter_notebooks/base/base.bit"

# Parse the design without reprogramming: boot.py already downloaded base.bit at
# boot, and re-downloading would reset the AD9361 datapath out from under the
# IIO drivers already bound to it.
try:
    ol = Overlay(BIT, download=False)
    print("bitstream on record :", os.path.basename(PL.bitfile_name))
    print("PL timestamp        :", PL.timestamp)
    print()
    print("IP blocks in the design:")
    for name in sorted(ol.ip_dict):
        print("   ", name)
except RuntimeError as e:
    # "No Devices Found" means XRT was not sourced in this process. Jupyter's
    # own environment sources /etc/profile.d/*.sh, so this only bites when
    # running the notebook from a bare shell (e.g. `sudo jupyter nbconvert`).
    print("PYNQ device discovery unavailable:", e)
    print("If you see this inside Jupyter, XRT is not sourced -- check")
    print("/etc/profile.d/ and that zocl is loaded (ls /dev/dri).")

## 2. The AD9361 is visible to libiio

Four IIO devices should appear: the Zynq `xadc`, the AD9361 PHY (SPI control),
and the receive / transmit datapath cores that live in the PL.

In [ ]:
import iio

ctx = iio.Context("local:")
print("libiio", iio.version, "| context:", ctx.name)
for dev in ctx.devices:
    name = dev.name or f"(unnamed id={dev.id})"
    kind = []
    if any(c.output for c in dev.channels):     kind.append("has TX chans")
    if any(not c.output for c in dev.channels): kind.append("has RX chans")
    print(f"  {name:28s} {len(dev.channels):3d} channels   {', '.join(kind) or 'no channels'}")

## 3. Configure the radio

In [ ]:
import adi

LO_HZ     = int(2.40e9)     # same for TX and RX -- see the note at the top
FS_HZ     = int(30.72e6)    # AD9361 sample rate
BW_HZ     = int(18e6)       # analog RF bandwidth
TONE_HZ   = int(2.00e6)     # baseband offset of the transmitted tone
NSAMP     = 1 << 14

sdr = adi.ad9361(uri="local:")

# --- radio ON -------------------------------------------------------------
# The board is normally parked in ALERT to keep it cool: in ALERT the PLLs stay
# locked but the TX/RX signal paths are powered down, which measured 27 C cooler
# on the AD9361 die and 7 C cooler on the Zynq. Nothing will transmit or receive
# until we put it back into FDD. The last cell parks it again.
ensm_on_entry = sdr._ctrl.attrs["ensm_mode"].value
sdr._ctrl.attrs["ensm_mode"].value = "fdd"
print(f"ensm_mode: {ensm_on_entry} -> {sdr._ctrl.attrs['ensm_mode'].value}  (radio on)")

sdr.sample_rate          = FS_HZ
sdr.rx_lo                = LO_HZ
sdr.tx_lo                = LO_HZ
sdr.rx_rf_bandwidth      = BW_HZ
sdr.tx_rf_bandwidth      = BW_HZ

sdr.rx_enabled_channels  = [0]
sdr.tx_enabled_channels  = [0]
sdr.rx_buffer_size       = NSAMP

# Manual gain: with a wired loopback the AGC would ride the level and make the
# measured power meaningless.
sdr.gain_control_mode_chan0 = "manual"
sdr.rx_hardwaregain_chan0   = 20.0     # dB
sdr.tx_hardwaregain_chan0   = -20.0    # dB of attenuation (0 = max output)

print(f"sample rate  : {sdr.sample_rate/1e6:.3f} MSPS")
print(f"RX LO        : {sdr.rx_lo/1e9:.6f} GHz")
print(f"TX LO        : {sdr.tx_lo/1e9:.6f} GHz")
print(f"RX bandwidth : {sdr.rx_rf_bandwidth/1e6:.3f} MHz")
print(f"RX gain      : {sdr.rx_hardwaregain_chan0} dB  (mode: {sdr.gain_control_mode_chan0})")
print(f"TX atten     : {sdr.tx_hardwaregain_chan0} dB")
assert sdr.rx_lo == sdr.tx_lo, "TX and RX LO must match for this loopback demo"

## 4. Transmit a tone

A cyclic buffer makes the hardware replay the same block forever, so the
transmitter runs continuously while we capture.

In [ ]:
import numpy as np

n = np.arange(NSAMP)
# Full-scale is +/-2**15; back off to 2**14 to leave headroom and avoid clipping
# in the interpolation filters.
tone = (0.5 * (2**15) * np.exp(2j * np.pi * TONE_HZ * n / FS_HZ)).astype(np.complex64)

sdr.tx_cyclic_buffer = True
sdr.tx_destroy_buffer()
sdr.tx(tone)

print(f"transmitting {TONE_HZ/1e6:.3f} MHz tone, {NSAMP} samples, cyclic")
print(f"  peak |amplitude| = {np.abs(tone).max():.0f} of 32768 full scale")

## 5. Capture the receiver

In [ ]:
import time

# Discard a few buffers so the AGC/filters settle and we are not looking at
# samples captured before the transmitter started.
for _ in range(4):
    sdr.rx()
time.sleep(0.1)
rx = sdr.rx()

print(f"captured {len(rx)} samples, dtype {rx.dtype}")
print(f"  mean |sample|    = {np.abs(rx).mean():8.1f}")
print(f"  peak |sample|    = {np.abs(rx).max():8.1f}  (clipping at ~2048 for 12-bit)")
print(f"  DC offset        = {rx.mean():.1f}")
assert np.abs(rx).mean() > 1.0, "receiver is flatlined -- is the loopback cable connected?"

## 6. Find the tone

The peak should sit at the transmitted offset. Its height above the noise floor
is the loopback SNR.

In [ ]:
win  = np.hanning(len(rx))
spec = np.fft.fftshift(np.fft.fft(rx * win))
psd  = 20 * np.log10(np.abs(spec) + 1e-12)
freq = np.fft.fftshift(np.fft.fftfreq(len(rx), 1/FS_HZ))

k         = int(np.argmax(psd))
f_peak    = freq[k]
noise     = np.median(psd)
snr_db    = psd[k] - noise
err_hz    = f_peak - TONE_HZ
bin_hz    = FS_HZ / len(rx)

print(f"expected tone : {TONE_HZ/1e6:+.4f} MHz")
print(f"measured peak : {f_peak/1e6:+.4f} MHz   (error {err_hz/1e3:+.1f} kHz, bin = {bin_hz/1e3:.1f} kHz)")
print(f"peak          : {psd[k]:.1f} dB (arbitrary ref)")
print(f"noise floor   : {noise:.1f} dB  ->  SNR {snr_db:.1f} dB")

assert abs(err_hz) < 5 * bin_hz, "peak is not where the tone was transmitted"
assert snr_db > 20, f"tone only {snr_db:.1f} dB above the floor -- check the attenuator/cabling"
print()
print("PASS: the transmitted tone was received through the loopback.")

## 7. Plots

In [ ]:
# Render inline. This works both interactively and under `jupyter nbconvert
# --execute`, which embeds the PNGs into the output notebook -- unlike the "Agg"
# backend, which draws to a file only and leaves plt.show() silent.
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(10, 7))

ax[0].plot(freq/1e6, psd, lw=0.8)
ax[0].axvline(TONE_HZ/1e6, color="tab:red", ls="--", lw=1, label=f"TX tone {TONE_HZ/1e6:.2f} MHz")
ax[0].plot(f_peak/1e6, psd[k], "o", color="tab:orange", label=f"peak ({snr_db:.0f} dB SNR)")
ax[0].set_xlabel("baseband frequency (MHz)")
ax[0].set_ylabel("magnitude (dB)")
ax[0].set_title(f"RX spectrum \u2014 TX/RX loopback via 10 dB pad, LO {LO_HZ/1e9:.2f} GHz")
ax[0].legend(loc="upper right")
ax[0].grid(alpha=0.3)

ns = 400
t_us = np.arange(ns) / FS_HZ * 1e6
ax[1].plot(t_us, rx[:ns].real, lw=0.9, label="I")
ax[1].plot(t_us, rx[:ns].imag, lw=0.9, label="Q")
ax[1].set_xlabel("time (\u00b5s)")
ax[1].set_ylabel("ADC counts")
ax[1].set_title(f"First {ns} received samples \u2014 expect a {1e6/TONE_HZ:.3f} \u00b5s period")
ax[1].legend(loc="upper right")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/home/xilinx/loopback_demo.png", dpi=110)   # keep a copy on disk too
plt.show()

## 8. Does the level match the 10 dB pad?

Sweeping TX attenuation should move the received level one-for-one. That is a
sanity check on the whole analog chain rather than a calibrated measurement —
the absolute offset also absorbs RX gain, cable loss and ADC scaling.

In [ ]:
rows = []
for atten in [-30.0, -25.0, -20.0, -15.0, -10.0]:
    sdr.tx_hardwaregain_chan0 = atten
    time.sleep(0.15)
    for _ in range(3):
        sdr.rx()
    r = sdr.rx()
    s = np.fft.fftshift(np.fft.fft(r * np.hanning(len(r))))
    p = 20*np.log10(np.abs(s[int(np.argmax(np.abs(s)))]) + 1e-12)
    rows.append((atten, p))

print(f"{'TX atten (dB)':>14}  {'RX peak (dB)':>13}  {'delta':>7}")
base = rows[0][1]
for atten, p in rows:
    print(f"{atten:>14.1f}  {p:>13.1f}  {p-base:>+7.1f}")

span_tx = rows[-1][0] - rows[0][0]
span_rx = rows[-1][1] - rows[0][1]
print()
print(f"TX attenuation swept {span_tx:+.0f} dB, RX level moved {span_rx:+.1f} dB")
if abs(span_rx - span_tx) < 6:
    print("-> tracks the commanded attenuation; the analog path is behaving linearly.")
else:
    print("-> does NOT track: suspect compression, AGC still active, or a bad connection.")

sdr.tx_hardwaregain_chan0 = -20.0
fig, ax = plt.subplots(figsize=(7, 4))
at = [r[0] for r in rows]; pk = [r[1] for r in rows]
ax.plot(at, pk, "o-", label="measured RX peak")
ideal = [pk[0] + (a - at[0]) for a in at]
ax.plot(at, ideal, "--", color="grey", label="ideal 1:1 tracking")
ax.set_xlabel("commanded TX attenuation (dB)")
ax.set_ylabel("RX peak (dB, arbitrary ref)")
ax.set_title("Does the received level track the commanded attenuation?")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Stop transmitting

In [ ]:
# --- radio OFF ------------------------------------------------------------
# Park the transceiver so the board is not cooking while idle. ALERT keeps the
# PLLs locked, so the next run returns to FDD without a re-tune.
def die_temps():
    x = "/sys/bus/iio/devices/iio:device0"
    raw   = int(open(f"{x}/in_temp0_raw").read())
    off   = int(open(f"{x}/in_temp0_offset").read())
    scale = float(open(f"{x}/in_temp0_scale").read())
    zynq  = (raw + off) * scale / 1000.0
    ad    = int(open("/sys/bus/iio/devices/iio:device1/in_temp0_input").read()) / 1000.0
    return zynq, ad

z0, a0 = die_temps()
print(f"before: zynq {z0:.1f} C, ad9361 {a0:.1f} C")

sdr.tx_destroy_buffer()
sdr.tx_hardwaregain_chan0 = -89.75          # AD9361 maximum attenuation
try:
    sdr.tx_hardwaregain_chan1 = -89.75      # ch1 defaults to -10 dB; mute it too
except Exception:
    pass
sdr._ctrl.attrs["ensm_mode"].value = "alert"
print("ensm_mode now:", sdr._ctrl.attrs["ensm_mode"].value, "(radio off)")

del sdr

# The die takes a few minutes to fall; this is only the immediate reading.
import time
time.sleep(20)
z1, a1 = die_temps()
print(f"after 20 s: zynq {z1:.1f} C ({z1-z0:+.1f}), ad9361 {a1:.1f} C ({a1-a0:+.1f})")
print()
print("Radio parked. Re-run the configure cell to switch it back on.")